# InsightAid AI — Forecasting Model Training (Phase 7, FYP2.md §50-51)

**Status: scaffolding.** No real sales export ships with this repo. Every
cell below is real, runnable code — the pipeline's pure-logic and live-model
modules (ARIMA, Prophet, XGBoost) are unit-tested (52/52 passing in this
repo's dev environment) — but until you point it at a real daily series
(see `../data/schema.md`), it produces nothing. Do not hand-write numbers
into a cell's output and keep them.

**Baselines:** simple moving average, ARIMA. **Main model:** Prophet, run
twice — sales-only (Model A, §50) and sentiment-enhanced (Model B, §50:
sentiment score, negative ratio, comment volume as regressors). **ML
comparison:** XGBoost. **LSTM:** only if `should_train_lstm()` says the
data justifies it (730+ daily observations) — see `../src/lstm_model.py`.

**Evaluation is time-aware rolling-origin, never a random split** (§51):
every candidate is scored on the exact same folds, so MAE/RMSE/MAPE are
directly comparable. **Never hardcoded which model wins** — `select_best()`
picks the real lowest-MAPE row.


## 1. Setup

In [ ]:
# Colab: uncomment to install (includes Prophet — confirmed to install
# and fit cleanly on both this repo's dev machine and Colab's Linux image).
# !pip install -q -r ../requirements.txt


In [ ]:
import sys
from pathlib import Path

REPO_ROOT_CANDIDATES = [Path.cwd().parent, Path.cwd()]
for candidate in REPO_ROOT_CANDIDATES:
    if (candidate / "src" / "data_loader.py").exists():
        sys.path.insert(0, str(candidate))
        break

import pandas as pd

from src.data_loader import build_daily_series
from src.evaluate import run_comparison, write_comparison
from src.lstm_model import should_train_lstm


## 2. Load data

Set `PROCESSED_DATA_CSV` to a real export of `processed_data` (see
`../data/schema.md`) before running anything below. `DAILY_AGGREGATES_CSV`
is optional — without it, only the sales-only variants can run
meaningfully (the sentiment-enhanced Prophet variant needs it).


In [ ]:
PROCESSED_DATA_CSV = "REPLACE_WITH_REAL_EXPORT_PATH.csv"  # see ../data/schema.md
DAILY_AGGREGATES_CSV = None  # optional: export of daily_aggregates, same schema

assert Path(PROCESSED_DATA_CSV).exists(), (
    "No sales data at PROCESSED_DATA_CSV. This notebook is scaffolding until "
    "you point it at a real export of processed_data (see ../data/schema.md). "
    "Do not fabricate rows to make this cell pass."
)

processed_df = pd.read_csv(PROCESSED_DATA_CSV)
daily_aggregates_df = pd.read_csv(DAILY_AGGREGATES_CSV) if DAILY_AGGREGATES_CSV else None

daily_df = build_daily_series(processed_df, daily_aggregates_df)
print(f"{len(daily_df)} daily observations, {daily_df['date'].min()} to {daily_df['date'].max()}")


## 3. LSTM data-volume check

Reports the real decision for this dataset — not a guess made ahead of
time. See `../src/lstm_model.py`'s docstring for the reasoning behind the
threshold.


In [ ]:
lstm_ok, lstm_reason = should_train_lstm(len(daily_df))
print(lstm_reason)


## 4. Run the full comparison

Runs moving-average, ARIMA, both Prophet variants, XGBoost, and (if step 3
said yes) LSTM, all on identical rolling-origin folds, then picks the real
best model by MAPE (ties on RMSE).


In [ ]:
HORIZON_DAYS = 30  # adjust to your actual forecasting need
RESULTS_DIR = Path("../results")

result = run_comparison(daily_df, horizon=HORIZON_DAYS)

for row in result.rows:
    marker = " <- BEST" if row.is_best else ""
    print(f"{row.model_type:15s} {row.variant:18s} MAE={row.mae:8.2f} RMSE={row.rmse:8.2f} MAPE={row.mape}{marker}")


## 5. Check against the target

The user's target is MAPE < 15% for the winning model. This is a **target,
not a guaranteed result** (FYP2.md §47's same framing, applied here) — the
cell below reports honestly whether it was met, it doesn't assert it.


In [ ]:
best_row = next(row for row in result.rows if row.is_best)
if best_row.mape is None:
    print("Best model's MAPE is undefined (every actual in the evaluation window was 0) - cannot compare to the 15% target.")
elif best_row.mape < 15:
    print(f"Target met: {best_row.model_type} ({best_row.variant}) MAPE = {best_row.mape}% < 15%")
else:
    print(f"Target NOT met: {best_row.model_type} ({best_row.variant}) MAPE = {best_row.mape}% >= 15%")


## 6. Save results

Writes a real `forecast_results_<run_id>.json` to `../results/`.


In [ ]:
out_path = write_comparison(result, RESULTS_DIR, run_id=result.rows[0].run_id)
print(f"wrote {out_path}")


## 7. Upload to Supabase

`forecast_results` (`supabase/migrations/0005_forecast_results.sql`) is
per-user data with standard RLS (`auth.uid() = user_id`) — unlike
`model_versions`, any authenticated user can write their OWN rows, so this
step can use the same Supabase client/credentials the app itself would use
for that user, not necessarily a service-role key. Paste your own
credentials into a Colab secret (never into this notebook's source).


In [ ]:
from dataclasses import asdict

# Example (uncomment and fill in your own values):
# from supabase import create_client
# supabase = create_client(SUPABASE_URL, SUPABASE_ANON_KEY)
# supabase.auth.sign_in_with_password({"email": ..., "password": ...})
# for row in result.rows:
#     supabase.table("forecast_results").insert(asdict(row)).execute()

for row in result.rows:
    print(row.model_type, row.variant, "is_best=" + str(row.is_best))
